**Table of contents**<a id='toc0_'></a>    
- 1. [简介](#toc1_)    
  - 1.1. [html](#toc1_1_)    
    - 1.1.1. [Header](#toc1_1_1_)    
    - 1.1.2. [body](#toc1_1_2_)    
  - 1.2. [config_json](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[简介](#toc0_)

## 1.1. <a id='toc1_1_'></a>[html](#toc0_)

### 1.1.1. <a id='toc1_1_1_'></a>[Header](#toc0_)

```html
<!-- PyScript -->
<link rel="stylesheet" href="https://pyscript.net/releases/2025.8.1/core.css">
<script type="module" src="https://pyscript.net/releases/2025.8.1/core.js"></script>
```

### 1.1.2. <a id='toc1_1_2_'></a>[body](#toc0_)

```html
<script type='py' src='learning_pyscript.py' config='./config.json'></script>
<script type='mpy' src='learning_pyscript.py' config='./config.json'></script>
```

## 1.2. <a id='toc1_2_'></a>[config_json](#toc0_)

```python
{
    "packages": ["numpy", "pandas", "matplotlib", "scipy", "sklearn", "requests"]
}

## python

```python
import numpy as np
```

# 访问与修改 DOM

PyScript（基于 Pyodide）在浏览器里运行 Python，与网页（DOM）交互有两条主要途径：

- 低阶 / 直接 JS 互操作：通过 js 全局命名空间（或 from js import ...）直接访问浏览器的 window、document、第三方 JS 变量等，行为和 JS 基本一致。

- PyScript 提供的高阶帮助器：pyscript 模块里有便捷对象/函数（如 pyscript.document, pyscript.display, Element 抽象等），以及 HTML 上的 py-* 属性（例如 py-click）作为快捷方式。

>这两条路都常用：直接 js 更原生、灵活；pyscript 的高阶 API 用起来更 Pythonic。

### DOM基本概念

#### DOM树是什么？

DOM（Document Object Model，文档对象模型）是浏览器将 HTML 或 XML 文档解析后形成的一种对象化表示。


例如：
```html
<div id="box">Hello <span>World</span></div>
```

```bash
Document
 └─ html
     └─ body
         └─ div#box
             ├─ Text("Hello ")
             └─ span
                 └─ Text("World")
```

>这棵树允许开发者通过 JavaScript 操作文档，从而实现动态网页。

| 节点类型         | 描述                                             |
| ------------ | ---------------------------------------------- |
| **Document** | 文档节点，DOM 树的根节点，对应整个 HTML 文档。可以用 `document` 访问。 |
| **Element**  | 元素节点，对应 HTML 标签。可以访问标签名、属性、子节点等。               |
| **Text**     | 文本节点，元素内部的文字。文本节点是元素的子节点，但不是元素本身。              |


### 操作 DOM 的基本动作

#### 查找节点

In [ ]:
document.getElementById('box');           // 通过 ID 查找
document.getElementsByClassName('item');  // 通过类名查找（返回 HTMLCollection）
document.querySelector('.item');          // CSS 选择器
document.querySelectorAll('.item');       // 返回所有匹配的节点（NodeList）

#### 读取节点

In [ ]:
const div = document.getElementById('box');
console.log(div.textContent);     // "Hello World"（文本内容）
console.log(div.innerHTML);       // "Hello <span>World</span>"（HTML 内容）
console.log(div.id);              // "box"

#### 创建节点

In [ ]:
const p = document.createElement('p');        // 创建 <p> 元素
const text = document.createTextNode('Hi');   // 创建文本节点
p.appendChild(text);                          // 将文本放入 <p>
document.body.appendChild(p);                 // 将 <p> 添加到页面

#### 修改节点

In [ ]:
const span = document.querySelector('span');
span.textContent = 'Everyone';   // 改变文本
span.setAttribute('class', 'highlight'); // 修改属性
span.style.color = 'red';        // 修改样式

#### 删除节点

In [ ]:
const div = document.getElementById('box');
div.parentNode.removeChild(div); // 删除节点
// 或现代写法：
div.remove();

#### 监听事件

In [ ]:
const button = document.querySelector('button');
button.addEventListener('click', () => {
    alert('按钮被点击了');
});

#### 总结

- DOM 是 HTML 的对象化表示，把文档转成一棵对象树。

- 节点类型：Document、Element、Text（最常用）。

- 基本操作：查找、读取、创建、修改、删除、监听事件。

- DOM 操作的本质：让页面可以动态地响应用户行为和数据变化。

## import js

js 是 Pyodide/浏览器环境中自动暴露的模块，允许你访问全局的 JavaScript 对象。

方法 B — 使用 js 命名空间（直接操作原生 DOM）。使用 from js import document 可以直接调用 document.createElement、querySelector、classList 等原生 API（返回的是 JS proxy 对象）。这一方式最接近 JS 的写法，性能与兼容性通常最好

In [ ]:
<div id="list"></div>

<py-script>
    # 从 js 导入 document、console
    from js import document, console

    # 创建元素并追加
    ul = document.createElement("ul")
    for i in range(3):
        li = document.createElement("li")
        li.innerText = f"条目 {i+1}"
        ul.appendChild(li)

    document.getElementById("list").appendChild(ul)
    console.log("已创建列表")
</py-script>


## pyscript的高阶API

方法 A — 使用 pyscript 高阶帮助器（更“Pythonic”）。pyscript 模块中包含 document, display, window, when 等便捷对象/函数。使用 display 会智能根据对象类型渲染。

In [ ]:
<div id="out"></div>

<py-script>
    from pyscript import document, display

    # 直接像 JS 一样调用 DOM API（pyscript.document 是 document 的代理）
    el = document.getElementById("out")
    el.innerHTML = "<b>这是通过 pyscript.document 设置的内容</b>"

    # 或使用 pyscript.display 将对象/图像/fig 渲染到页面
    display("Hello from display()", target="out")
</py-script>


# 事件处理

事件处理（click、input……）— 正确方法与坑。

直接把 Python 函数作为 JS 回调会遇到“回调被垃圾回收 / 生命周期”问题。正确做法是用 pyodide 的 create_proxy 或 pyscript 封装的同类工具，它会把 Python 可调用对象包装成 JS 可调用对象并保持引用。

示例（用 create_proxy）：

In [ ]:
<button id="btn">点我</button>
<div id="counter">0</div>

<py-script>
    from js import document
    from pyodide.ffi import create_proxy  # 或 pyscript.ffi.create_proxy

    count = 0
    def on_click(ev):
        global count
        count += 1
        document.getElementById("counter").innerHTML = str(count)

    # wrap 并保持 proxy 引用（否则会被回收）
    proxy = create_proxy(on_click)
    document.getElementById("btn").addEventListener("click", proxy)

    # 当不需要监听时，移除并销毁 proxy:
    # document.getElementById("btn").removeEventListener("click", proxy)
    # proxy.destroy()
</py-script>


>替代方式：如果你更习惯 HTML 属性形式，PyScript 支持 py-click="handler" 这种快捷写法（将 HTML 的事件直接映射到 Python 函数名），这对简单交互很方便。

# 异步请求（fetch / AJAX）：Python 风格的 fetch

PyScript 提供了 pyscript.fetch（也可直接用 js.fetch），可在 Python 中 await 使用，非常像浏览器的 Fetch API：

In [ ]:
<div id="data"></div>

<py-script>
    from pyscript import fetch
    from js import document

    async def load_json():
        resp = await fetch("data/publications.json")
        data = await resp.json()
        out = document.getElementById("data")
        for item in data:
            out.innerHTML += f"<p>{item['title']} ({item['year']})</p>"

    await load_json()
</py-script>


>注意 CORS：向外部 API 请求时可能遇到跨域问题（服务器需返回 Access-Control-Allow-Origin），这和普通前端开发完全相同。

# Python ↔ JavaScript 数据互通（类型转换与 Proxy）

- 当 Python 把对象传给 JS 时，Pyodide 会生成 PyProxy；反过来则生成 JsProxy。你可以用 pyodide.to_js() / jsproxy.to_py() 等显式转换来控制行为。

- 对常见简单数据（数字、字符串、dict/list）通常自动转换良好；复杂对象可以用 to_js() 指定深度/代理策略。
pyodide.org
+1

示例（把 Python 列表传给 JS）

In [ ]:
from pyodide import to_js


js_array = to_js([1,2,3])   # 现在可以作为 JS 数组使用